# 08 — Joint Training: Detection + Lane Segmentation

**Goal:** Train a shared YOLO26 backbone/neck with both the native detection head and a lightweight lane segmentation head in a single stage.

## Architecture
```
Input → YOLO26 Backbone → Neck (FPN/PAN) → ┬→ Detect Head (10-class BDD boxes)
                                            └→ Lane Seg Head (binary mask)
```

## Colab Notes
- **GPU required:** T4 minimum (recommended: A100)
- **Training time:** ~2-4 hours for 10 epochs on full BDD100K with T4
- **Memory:** ~12+ GB GPU for yolo26s at batch=8, imgsz=640
- Uses **mixed precision (AMP)** for speed
- Uses **differential learning rates**: lower for pretrained backbone, higher for new lane head

### Prerequisites
Run these notebooks first:
- `02_bdd100k_preparation.ipynb` (detection labels)
- `07_prepare_lane_masks.ipynb` (lane masks)

## 1 — Setup

In [ ]:
!pip install -q ultralytics>=8.4.0 opencv-python matplotlib pyyaml tqdm

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"✅ GPU: {gpu} ({mem:.1f} GB)")
    if mem < 12:
        print("⚠️ Low GPU RAM. Reduce batch_size to 4 or use yolo26n.")
else:
    print("❌ No GPU! Joint training requires a GPU.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, yaml

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
DATASET_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo")

# Add project src to path
PROJECT_SRC = os.path.join(ECOCAR_ROOT, "project", "src")
if os.path.isdir(PROJECT_SRC):
    sys.path.insert(0, os.path.dirname(PROJECT_SRC))

# Debug mode
DEBUG_LIMIT = None  # Set to 100 for quick test

if DEBUG_LIMIT:
    print(f"⚡ DEBUG MODE: {DEBUG_LIMIT} samples")

## 2 — Training Configuration

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
CONFIG = {
    # Model
    'pretrained_weights': 'yolo26s.pt',  # or yolo26n.pt for less memory
    'lane_hidden_channels': 64,
    'mask_height': 180,
    'mask_width': 320,
    
    # Dataset
    'train_images': os.path.join(DATASET_DIR, 'images', 'train'),
    'train_labels': os.path.join(DATASET_DIR, 'labels', 'train'),
    'train_masks':  os.path.join(DATASET_DIR, 'masks', 'train'),
    'val_images':   os.path.join(DATASET_DIR, 'images', 'val'),
    'val_labels':   os.path.join(DATASET_DIR, 'labels', 'val'),
    'val_masks':    os.path.join(DATASET_DIR, 'masks', 'val'),
    
    # Training
    'epochs': 10,
    'batch_size': 8,
    'img_size': 640,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    
    # Loss weights: α·det + β·lane
    'det_loss_weight': 1.0,
    'lane_loss_weight': 0.5,
    
    # Performance
    'amp': True,
    'num_workers': 4,
    'save_period': 5,
    
    # Output
    'save_dir': os.path.join(ECOCAR_ROOT, 'training_runs', 'joint_det_lane'),
}

print("Joint Training Config:")
for k, v in CONFIG.items():
    print(f"  {k:<25} = {v}")

## 3 — Build Multi-Task Model

In [ ]:
try:
    from src.multitask_model import MultiTaskYOLO, build_multitask_model
    print("✅ Imported MultiTaskYOLO from src")
except ImportError:
    print("⚠️ Could not import from src, using inline definition")
    # See src/multitask_model.py for the full implementation
    # The inline version is abbreviated — copy multitask_model.py to Drive for best results
    raise ImportError(
        "Please upload the 'src/' folder to your Google Drive at "
        f"{os.path.dirname(PROJECT_SRC)}\n"
        "The multi-task model is too complex for inline definition."
    )

In [ ]:
# Build the multi-task model
model = build_multitask_model(
    weights=CONFIG['pretrained_weights'],
    mask_height=CONFIG['mask_height'],
    mask_width=CONFIG['mask_width'],
    lane_hidden_channels=CONFIG['lane_hidden_channels'],
)

# Print parameter counts
total = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for n, p in model.named_parameters() if 'lane_head' not in n and 'detect' not in n)
det_params = sum(p.numel() for n, p in model.named_parameters() if 'detect' in n)
lane_params = sum(p.numel() for p in model.lane_head.parameters())

print(f"\nParameter counts:")
print(f"  Backbone+Neck: {backbone_params:>12,}")
print(f"  Detect head:   {det_params:>12,}")
print(f"  Lane head:     {lane_params:>12,}")
print(f"  Total:         {total:>12,}")

## 4 — Create Joint Dataset

In [ ]:
try:
    from src.joint_dataset import JointBDDDataset, joint_collate_fn
    print("✅ Imported JointBDDDataset from src")
except ImportError:
    raise ImportError("Please upload src/ to Drive. See instructions above.")

In [ ]:
from torch.utils.data import DataLoader

# Train dataset
train_dataset = JointBDDDataset(
    images_dir=CONFIG['train_images'],
    labels_dir=CONFIG['train_labels'],
    masks_dir=CONFIG['train_masks'],
    img_size=CONFIG['img_size'],
    mask_height=CONFIG['mask_height'],
    mask_width=CONFIG['mask_width'],
    augment=True,
    debug_limit=DEBUG_LIMIT,
)

# Val dataset
val_dataset = JointBDDDataset(
    images_dir=CONFIG['val_images'],
    labels_dir=CONFIG['val_labels'],
    masks_dir=CONFIG['val_masks'],
    img_size=CONFIG['img_size'],
    mask_height=CONFIG['mask_height'],
    mask_width=CONFIG['mask_width'],
    augment=False,
    debug_limit=DEBUG_LIMIT,
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    collate_fn=joint_collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    collate_fn=joint_collate_fn,
    pin_memory=True,
)

print(f"\nTrain: {len(train_dataset)} samples → {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples → {len(val_loader)} batches")

## 5 — Sanity Check Batch

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

# Get one batch
batch = next(iter(train_loader))

print(f"Images shape:     {batch['images'].shape}")
print(f"Det targets shape:{batch['det_targets'].shape}")
print(f"Lane masks shape: {batch['lane_masks'].shape}")

# Show first 4 images with their lane masks
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i in range(min(4, batch['images'].shape[0])):
    img = batch['images'][i].permute(1, 2, 0).numpy()
    mask = batch['lane_masks'][i, 0].numpy()
    
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image {i}', fontsize=9)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(mask, cmap='gray')
    axes[1, i].set_title(f'Lane mask {i}', fontsize=9)
    axes[1, i].axis('off')

plt.suptitle('Sanity Check — Training Batch', fontsize=12)
plt.tight_layout()
plt.show()

## 6 — Train!

In [ ]:
try:
    from src.joint_trainer import JointTrainer, plot_joint_training_history
    print("✅ Imported JointTrainer from src")
except ImportError:
    raise ImportError("Please upload src/ to Drive.")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

trainer = JointTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    det_loss_weight=CONFIG['det_loss_weight'],
    lane_loss_weight=CONFIG['lane_loss_weight'],
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
    device=device,
    amp=CONFIG['amp'],
    save_dir=CONFIG['save_dir'],
    save_period=CONFIG['save_period'],
)

In [ ]:
# Start training
history = trainer.train(epochs=CONFIG['epochs'])

## 7 — Training Results

In [ ]:
# Plot training curves
save_path = os.path.join(CONFIG['save_dir'], 'training_curves.png')
plot_joint_training_history(history, save_path=save_path)

In [ ]:
# Show saved weights
weights_dir = os.path.join(CONFIG['save_dir'], 'weights')
if os.path.isdir(weights_dir):
    print("\nSaved weights:")
    for f in sorted(os.listdir(weights_dir)):
        size_mb = os.path.getsize(os.path.join(weights_dir, f)) / (1024**2)
        print(f"  {f} ({size_mb:.1f} MB)")

## 8 — Backup Best Weights

In [ ]:
import shutil

best_src = os.path.join(weights_dir, 'best.pt')
backup_dir = os.path.join(ECOCAR_ROOT, 'weights')
os.makedirs(backup_dir, exist_ok=True)

if os.path.isfile(best_src):
    dst = os.path.join(backup_dir, 'joint_det_lane_best.pt')
    shutil.copy2(best_src, dst)
    print(f"✅ Best weights backed up: {dst}")
    print(f"\n📌 Use this path in notebook 09:")
    print(f"   {dst}")

In [ ]:
print("\n" + "="*60)
print(" JOINT TRAINING — COMPLETE")
print("="*60)
print(f" Model:          {CONFIG['pretrained_weights']} + LaneSegHead")
print(f" Epochs:         {CONFIG['epochs']}")
print(f" Det loss wt:    {CONFIG['det_loss_weight']}")
print(f" Lane loss wt:   {CONFIG['lane_loss_weight']}")
print(f" Outputs:        {CONFIG['save_dir']}")
print("="*60)
print("\n🎯 Proceed to notebook 09 for joint inference!")